In [ ]:
import h5py as hf
import os
import shutil

import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import imageio

In [ ]:
with hf.File("/work/stovey/ballistic-diamond/simulation/database.hdf5") as db:
    vel_data = db['1']["Velocities"][:, ::100]
    pos_data = db["1"]["Positions"][:, ::100]
    zn_pos = db['2']["Positions"][:, ::100]

In [ ]:
ke = 0.5 * (jnp.array(vel_data) ** 2).sum(axis=-1)

In [ ]:
ke_min = jnp.min(ke)
ke_max = jnp.max(ke)

In [ ]:
center = jnp.array([35.0, 35.0, 150.0])
distance_from_center = jnp.linalg.norm(pos_data - center, axis=-1)
distance_from_center = 1 / distance_from_center
distance_from_center /= distance_from_center.max()

In [ ]:
dir_name = "temp-frames"
os.mkdir("temp-frames")

frames = []

for frame in range(pos_data.shape[1]):

    fig = plt.figure()
    ax = fig.add_subplot(projection='3d')

    plt_pos = pos_data
    time=frame
    im = ax.scatter(
        pos_data[:, time, 0], 
        pos_data[:, time, 1], 
        pos_data[:, time, 2], 
        c=ke[:, time], 
        norm=LogNorm(vmin=ke_min, vmax=ke_max),
        cmap="hot",
        # alpha=distance_from_center[:, time],
        # alpha=ke[:, time]
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    ax.set_xlim(-1, 71)
    ax.set_ylim(-1, 71)
    ax.set_zlim(190, 510)

    plt.colorbar(im, label="Kinetic Energy")
    plt.savefig(f"{dir_name}/{frame}.png")
    frames.append(f"{dir_name}/{frame}.png")
    plt.close()

In [ ]:
# Create GIF
with imageio.get_writer('3d-collision.gif', mode='I', duration=0.1) as writer:
    for filename in frames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [ ]:
shutil.rmtree("temp-frames")

In [ ]:
dir_name = "temp-frames"
os.mkdir("temp-frames")
for frame in range(pos_data.shape[1]):

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    time = frame

    # xy
    ax[0].scatter(
        pos_data[:, time, 0], 
        pos_data[:, time, 1], 
        c=ke[:, time], 
        norm=LogNorm(vmin=ke_min, vmax=ke_max),
        cmap="hot",
        s=3 * np.log10(ke[:, time]) + 3,
        # alpha=ke[:, time]
    )
    ax[0].set_xlim(-5, 75)
    ax[0].set_ylim(-5, 75)
    ax[0].set_xlabel("x")
    ax[0].set_ylabel("y")
    ax[0].set_title("xy Projection")
    # xz
    ax[1].scatter(
        pos_data[:, time, 0], 
        pos_data[:, time, 2], 
        c=ke[:, time], 
        norm=LogNorm(vmin=ke_min, vmax=ke_max),
        cmap="hot",
        s=3 * np.log10(ke[:, time]) + 3,
        # alpha=ke[:, time]
    )
    ax[1].set_title("xz Projection")
    ax[1].set_xlim(-5, 75)
    ax[1].set_ylim(190, 535)
    ax[1].scatter(zn_pos[:, time, 0], zn_pos[:, time, 2])
    ax[1].set_xlabel("x")
    ax[1].set_ylabel("z")
    # yz
    im = ax[2].scatter(
        pos_data[:, time, 1], 
        pos_data[:, time, 2], 
        c=ke[:, time], 
        norm=LogNorm(vmin=ke_min, vmax=ke_max),
        cmap="hot",
        s=3 * np.log10(ke[:, time]) + 3,
        # alpha=ke[:, time]
    )
    ax[2].set_title("yz Projection")
    ax[2].set_xlabel("y")
    ax[2].set_ylabel("z")
    ax[2].set_xlim(-5, 75)
    ax[2].set_ylim(190, 535)
    ax[2].scatter(zn_pos[:, time, 1], zn_pos[:, time, 2])

    plt.colorbar(im, label="Kinetic Energy")
    plt.savefig(f"{dir_name}/{frame}.png")
    frames.append(f"{dir_name}/{frame}.png")
    plt.close()

In [ ]:
frames = [f'temp-frames/{i}.png' for i in range(pos_data.shape[1])]

In [ ]:
# Create GIF
with imageio.get_writer('projected-collision.gif', mode='I', duration=0.1) as writer:
    for filename in frames:
        image = imageio.imread(filename)
        writer.append_data(image)

In [ ]:
shutil.rmtree("temp-frames")